In [ ]:
import os
import sys
import subprocess
import importlib.util
import importlib.metadata
import site

if site.ENABLE_USER_SITE:
    site.addsitedir(site.getusersitepackages())

DEPENDENCIES = {'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.1,<3', 'scipy': 'scipy>=1.11,<2', 'sklearn': 'scikit-learn>=1.4,<2', 'matplotlib': 'matplotlib>=3.8,<4', 'joblib': 'joblib>=1.3,<2', 'threadpoolctl': 'threadpoolctl>=3.2,<4'}
if importlib.util.find_spec("packaging") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "packaging>=23,<27"])
from packaging.requirements import Requirement

missing_packages = []
for module, requirement_text in DEPENDENCIES.items():
    requirement = Requirement(requirement_text)
    try:
        installed_version = importlib.metadata.version(requirement.name)
    except importlib.metadata.PackageNotFoundError:
        installed_version = None
    if importlib.util.find_spec(module) is None or installed_version is None or installed_version not in requirement.specifier:
        missing_packages.append(requirement_text)
if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", *missing_packages])
importlib.invalidate_caches()
for thread_variable in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"]:
    os.environ.setdefault(thread_variable, "2")


In [ ]:
SEED = 42
SAMPLE_SIZE = 1200
OUTPUT_ROOT = os.environ.get("FABRIC_STARTER_OUTPUT", "")
WORKFLOW = "144_projected_frequent_sequence_mining"


In [ ]:
import json
import hashlib
import time
import uuid
import tempfile
from pathlib import Path
from datetime import datetime, timezone
from dataclasses import dataclass, asdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from threadpoolctl import threadpool_limits

thread_limits = threadpool_limits(limits=2)
rng = np.random.default_rng(SEED)
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + "_" + uuid.uuid4().hex[:8]
lakehouse_files = Path("/lakehouse/default/Files")
output_base = Path(OUTPUT_ROOT).expanduser() if OUTPUT_ROOT else (lakehouse_files / "fabric_python_starters" if lakehouse_files.is_dir() else Path.cwd() / "fabric_python_outputs")
output_dir = output_base / WORKFLOW / run_id
output_dir.mkdir(parents=True, exist_ok=True)
working_context = tempfile.TemporaryDirectory(prefix="fabric_starter_")
working_dir = Path(working_context.name)
run_started = time.perf_counter()
artifact_records = []

def json_value(value):
    if isinstance(value, (np.integer, np.floating, np.bool_)):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (Path, pd.Timestamp, datetime)):
        return str(value)
    raise TypeError(type(value).__name__)

def record_artifact(path):
    path = Path(path)
    artifact_records.append({"name": path.name, "bytes": path.stat().st_size, "sha256": hashlib.sha256(path.read_bytes()).hexdigest()})
    return path

def save_table(frame, name):
    path = output_dir / (name + ".csv")
    frame.to_csv(path, index=False)
    record_artifact(path)
    return path

def save_json(payload, name):
    path = output_dir / (name + ".json")
    path.write_text(json.dumps(payload, indent=2, default=json_value, allow_nan=False), encoding="utf-8")
    record_artifact(path)
    return path

def save_model(model, name):
    path = output_dir / (name + ".joblib")
    joblib.dump(model, path)
    record_artifact(path)
    return path

def save_figure(figure, name):
    figure.tight_layout()
    path = output_dir / (name + ".png")
    figure.savefig(path, dpi=130, bbox_inches="tight")
    record_artifact(path)
    plt.show()
    plt.close(figure)
    return path

def finish(metrics, tables=()):
    versions = {}
    for requirement in DEPENDENCIES.values():
        distribution = requirement.split(">=")[0]
        versions[distribution] = importlib.metadata.version(distribution)
    payload = {"workflow": WORKFLOW, "run_id": run_id, "seed": SEED, "python": sys.version.split()[0], "packages": versions, "storage": "lakehouse" if lakehouse_files in output_dir.parents else "local", "output_dir": str(output_dir), "elapsed_seconds": round(time.perf_counter() - run_started, 3), "metrics": metrics, "artifacts": list(artifact_records)}
    save_json(payload, "run_manifest")
    print(json.dumps(payload, indent=2, default=json_value, allow_nan=False))
    for table in tables:
        print(table.head(12).to_string(index=False))
    working_context.cleanup()
    return payload


In [ ]:
from collections import defaultdict

alphabet=['visit','search','compare','cart','purchase','support','return']
sequences=[]
for user in range(450):
    sequence=['visit']
    sequence+=rng.choice(alphabet[1:4],int(rng.integers(1,5))).tolist()
    if 'cart' in sequence and rng.random()<.7:
        sequence+=['purchase']
    if 'purchase' in sequence and rng.random()<.3:
        sequence+=['support']
    if 'support' in sequence and rng.random()<.25:
        sequence+=['return']
    sequences.append(sequence)
minimum_support=45
patterns=[]

def grow(prefix,projected):
    extensions=defaultdict(list)
    for sequence_id,suffix in projected:
        first={}
        for position,event in enumerate(suffix):
            first.setdefault(event,position)
        for event,position in first.items():
            extensions[event].append((sequence_id,suffix[position+1:]))
    for event,projection in sorted(extensions.items()):
        support=len(projection)
        if support<minimum_support:
            continue
        pattern=prefix+(event,)
        patterns.append({'pattern':pattern,'support_count':support,'length':len(pattern)})
        if len(pattern)<4:
            grow(pattern,projection)

def is_subsequence(pattern,sequence):
    position=0
    for event in sequence:
        if position<len(pattern) and pattern[position]==event:
            position+=1
    return position==len(pattern)


In [ ]:

grow((),[(index,sequence) for index,sequence in enumerate(sequences)])
for record in patterns:
    exact=sum(is_subsequence(record['pattern'],sequence) for sequence in sequences)
    assert exact==record['support_count']
lookup={record['pattern']:record['support_count'] for record in patterns}
rows=[]
for record in patterns:
    pattern=record['pattern']
    prefix_support=lookup.get(pattern[:-1],len(sequences))
    closed=not any(len(other)>len(pattern) and count==record['support_count'] and is_subsequence(pattern,other) for other,count in lookup.items())
    rows.append({'pattern':json.dumps(pattern),'length':len(pattern),'support_count':record['support_count'],'support':record['support_count']/len(sequences),'prefix_extension_confidence':record['support_count']/prefix_support,'closed_within_length_limit':closed})
output=pd.DataFrame(rows).sort_values(['length','support_count'],ascending=[False,False])
save_table(output,'frequent_ordered_subsequences')
save_table(pd.DataFrame({'sequence_id':range(len(sequences)),'events':[json.dumps(sequence) for sequence in sequences]}),'user_event_sequences')
save_table(output.loc[output.closed_within_length_limit],'closed_patterns_up_to_length_four')
fig,ax=plt.subplots(figsize=(8,4))
output.groupby('length').size().plot.bar(ax=ax)
ax.set(ylabel='Frequent patterns')
save_figure(fig,'sequential_pattern_lengths')
finish({'sequences':len(sequences),'patterns':len(output),'maximum_pattern_length':4,'all_support_counts_verified':True,'events_need_not_be_contiguous':True},[output.head(12)])
